# Flow Matching

## standalone implementation of flow matching

In [1]:
import torch
from torch import nn, optim
from sklearn.datasets import make_moons
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
class Flow(nn.Module):
    def __init__(self, dim: int = 2, hidden_dim: int = 64):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dim + 1, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.layers(torch.cat([t, x_t], dim=-1))

    def step(self, x_t: torch.Tensor, t_start: torch.Tensor, t_end: torch.Tensor) -> torch.Tensor:
        t_start = t_start.view(1, 1).expand(x_t.shape[0], 1)
        # use midpoint ODE solver for simplicity
        return x_t + (t_end - t_start) * self.forward(x_t + self.forward(x_t, t_start) * (t_end - t_start) / 2, t_start + (t_end - t_start) / 2)

In [3]:
flow = Flow()
optimizer = optim.Adam(flow.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

In [4]:
for _ in range(10000):
    x1 = torch.Tensor(make_moons(n_samples=256, noise=0.05)[0])
    x0 = torch.randn_like(x1)
    t = torch.rand(len(x1), 1)
    x_t = (1 - t) * x0 + t * x1
    dx_t = x1 - x0
    optimizer.zero_grad()
    loss_fn(flow(x_t, t), dx_t).backward()
    optimizer.step()

In [7]:
x = torch.randn(300, 2)
n_steps = 8
timesteps = torch.linspace(0, 1, n_steps + 1)
fig = make_subplots(rows=1, cols=n_steps + 1, subplot_titles=[f"t={t:.2f}" for t in timesteps], shared_yaxes=True)
fig.add_trace(go.Scatter(x=x.detach()[:, 0], y=x.detach()[:, 1], mode="markers"), row=1, col=1)
for i in range(1, n_steps + 1):
    x = flow.step(x, timesteps[i - 1], timesteps[i])
    fig.add_trace(go.Scatter(x=x.detach()[:, 0], y=x.detach()[:, 1], mode="markers"), row=1, col=i + 1)
fig.show()